# SAP Intelligent Document Processing and Query Answering Agent

## Overview
This notebook implements a real-time Retrieval Augmented Generation (RAG) system for SAP documents including:
- Invoices, Purchase Orders, and Reports
- Intelligent query answering with contextual information
- Document retrieval and processing
- Performance evaluation metrics

## Architecture Components:
1. **Document Collection & Preprocessing**: Load and clean SAP documents
2. **Embedding Generation**: Convert documents to vector representations
3. **Vector Database**: Store and retrieve relevant documents
4. **RAG Pipeline**: Retrieve context and generate answers
5. **LLM Integration**: Fine-tuned or prompt-engineered model for SAP domain
6. **Evaluation**: Metrics to assess system performance

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q langchain langchain-community langchain-openai
!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q openai
!pip install -q pypdf
!pip install -q python-dotenv
!pip install -q pandas numpy
!pip install -q faiss-cpu
!pip install -q tiktoken
!pip install -q ragas  # For evaluation metrics

In [ ]:
# Import necessary libraries
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import List, Dict, Any, Tuple
import warnings
warnings.filterwarnings('ignore')

# LangChain imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma, FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# For evaluation
from sklearn.metrics.pairwise import cosine_similarity

print("✓ All libraries imported successfully")

In [ ]:
# Configuration
class Config:
    """Configuration parameters for the SAP RAG system"""
    
    # API Keys (use environment variables in production)
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-api-key-here")
    
    # Model Configuration
    EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # Open-source embeddings
    LLM_MODEL = "gpt-4"  # or "gpt-3.5-turbo" for faster/cheaper inference
    TEMPERATURE = 0.1  # Low temperature for factual responses
    
    # Chunking parameters
    CHUNK_SIZE = 1000
    CHUNK_OVERLAP = 200
    
    # Retrieval parameters
    TOP_K_DOCUMENTS = 5  # Number of documents to retrieve
    
    # Vector store
    VECTOR_STORE_PATH = "./sap_vector_store"
    COLLECTION_NAME = "sap_documents"
    
    # Document types
    SUPPORTED_DOCUMENT_TYPES = ["invoice", "purchase_order", "sales_order", "report"]

config = Config()
print("✓ Configuration loaded")

## 2. SAP Document Generation (Simulated Data)

In production, you would load real SAP documents. Here we generate synthetic data for demonstration.

In [ ]:
class SAPDocumentGenerator:
    """Generate synthetic SAP documents for testing"""
    
    def __init__(self):
        self.companies = ["Acme Corp", "Global Tech Inc", "Enterprise Solutions Ltd", "MegaCorp Industries"]
        self.materials = [
            ("MAT-001", "Laptop Computer", 1200),
            ("MAT-002", "Office Desk", 450),
            ("MAT-003", "Software License", 599),
            ("MAT-004", "Printer", 350),
            ("MAT-005", "Monitor 27\"", 300)
        ]
        self.departments = ["IT", "Finance", "Operations", "HR", "Sales"]
    
    def generate_invoice(self, invoice_id: int) -> str:
        """Generate a synthetic SAP invoice"""
        company = np.random.choice(self.companies)
        date = datetime.now() - timedelta(days=np.random.randint(1, 365))
        items = np.random.randint(1, 5)
        
        invoice_text = f"""SAP INVOICE DOCUMENT
Invoice Number: INV-{invoice_id:06d}
Date: {date.strftime('%Y-%m-%d')}
Customer: {company}
Customer ID: CUST-{np.random.randint(1000, 9999)}
Payment Terms: Net 30 Days
Status: {'Paid' if np.random.random() > 0.3 else 'Outstanding'}

LINE ITEMS:
"""
        total = 0
        for i in range(items):
            mat_code, mat_name, unit_price = self.materials[np.random.randint(0, len(self.materials))]
            quantity = np.random.randint(1, 10)
            line_total = unit_price * quantity
            total += line_total
            invoice_text += f"{i+1}. {mat_code} - {mat_name} | Qty: {quantity} | Unit Price: ${unit_price} | Total: ${line_total}\n"
        
        tax = total * 0.08
        grand_total = total + tax
        
        invoice_text += f"\nSubtotal: ${total:.2f}\nTax (8%): ${tax:.2f}\nGrand Total: ${grand_total:.2f}\n"
        invoice_text += f"\nDue Date: {(date + timedelta(days=30)).strftime('%Y-%m-%d')}"
        
        return invoice_text
    
    def generate_purchase_order(self, po_id: int) -> str:
        """Generate a synthetic SAP purchase order"""
        vendor = np.random.choice(self.companies)
        date = datetime.now() - timedelta(days=np.random.randint(1, 180))
        department = np.random.choice(self.departments)
        
        po_text = f"""SAP PURCHASE ORDER
PO Number: PO-{po_id:06d}
Date: {date.strftime('%Y-%m-%d')}
Vendor: {vendor}
Vendor ID: VEN-{np.random.randint(1000, 9999)}
Requesting Department: {department}
Delivery Date: {(date + timedelta(days=14)).strftime('%Y-%m-%d')}
Status: {'Delivered' if np.random.random() > 0.4 else 'In Transit' if np.random.random() > 0.5 else 'Pending'}

ITEMS ORDERED:
"""
        items = np.random.randint(1, 4)
        total = 0
        for i in range(items):
            mat_code, mat_name, unit_price = self.materials[np.random.randint(0, len(self.materials))]
            quantity = np.random.randint(5, 20)
            line_total = unit_price * quantity
            total += line_total
            po_text += f"{i+1}. {mat_code} - {mat_name} | Quantity: {quantity} | Unit Price: ${unit_price} | Total: ${line_total}\n"
        
        po_text += f"\nTotal Order Value: ${total:.2f}"
        po_text += f"\nApproved By: Manager-{np.random.randint(100, 999)}"
        
        return po_text
    
    def generate_sales_order(self, so_id: int) -> str:
        """Generate a synthetic SAP sales order"""
        customer = np.random.choice(self.companies)
        date = datetime.now() - timedelta(days=np.random.randint(1, 90))
        
        so_text = f"""SAP SALES ORDER
Sales Order Number: SO-{so_id:06d}
Date: {date.strftime('%Y-%m-%d')}
Customer: {customer}
Customer ID: CUST-{np.random.randint(1000, 9999)}
Sales Representative: SR-{np.random.randint(100, 999)}
Order Status: {'Completed' if np.random.random() > 0.5 else 'Processing'}
Shipping Method: {'Express' if np.random.random() > 0.7 else 'Standard'}

ORDER DETAILS:
"""
        items = np.random.randint(1, 6)
        total = 0
        for i in range(items):
            mat_code, mat_name, unit_price = self.materials[np.random.randint(0, len(self.materials))]
            quantity = np.random.randint(1, 15)
            discount = np.random.choice([0, 5, 10, 15])
            line_total = unit_price * quantity * (1 - discount/100)
            total += line_total
            so_text += f"{i+1}. {mat_code} - {mat_name} | Qty: {quantity} | Price: ${unit_price} | Discount: {discount}% | Total: ${line_total:.2f}\n"
        
        so_text += f"\nTotal Amount: ${total:.2f}"
        so_text += f"\nExpected Delivery: {(date + timedelta(days=7)).strftime('%Y-%m-%d')}"
        
        return so_text
    
    def generate_report(self, report_id: int) -> str:
        """Generate a synthetic SAP financial report"""
        department = np.random.choice(self.departments)
        quarter = np.random.choice(["Q1", "Q2", "Q3", "Q4"])
        year = np.random.choice([2023, 2024])
        
        revenue = np.random.randint(500000, 2000000)
        expenses = int(revenue * np.random.uniform(0.6, 0.9))
        profit = revenue - expenses
        
        report_text = f"""SAP FINANCIAL REPORT
Report ID: REP-{report_id:06d}
Department: {department}
Period: {quarter} {year}
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}

FINANCIAL SUMMARY:
Total Revenue: ${revenue:,}
Total Expenses: ${expenses:,}
Net Profit: ${profit:,}
Profit Margin: {(profit/revenue*100):.2f}%

KEY METRICS:
- Number of Transactions: {np.random.randint(100, 500)}
- Average Transaction Value: ${revenue/np.random.randint(100, 500):,.2f}
- Customer Satisfaction: {np.random.randint(75, 98)}%
- On-time Delivery Rate: {np.random.randint(85, 99)}%

BUDGET VARIANCE:
Actual vs Budget: {np.random.choice(['+', '-'])}{np.random.randint(1, 15)}%

NOTES:
Performance {'exceeded' if profit > expenses * 0.2 else 'met'} expectations for this period.
Key growth drivers: {'Product sales' if np.random.random() > 0.5 else 'Service contracts'}
"""
        return report_text

# Generate sample documents
generator = SAPDocumentGenerator()
print("✓ SAP Document Generator initialized")

In [ ]:
# Generate a diverse set of SAP documents
def create_document_dataset(num_each: int = 25) -> List[Document]:
    """Create a dataset of various SAP documents"""
    documents = []
    
    print(f"Generating {num_each * 4} SAP documents...")
    
    # Generate invoices
    for i in range(num_each):
        text = generator.generate_invoice(i + 1)
        doc = Document(
            page_content=text,
            metadata={
                "document_type": "invoice",
                "document_id": f"INV-{i+1:06d}",
                "created_date": datetime.now().isoformat()
            }
        )
        documents.append(doc)
    
    # Generate purchase orders
    for i in range(num_each):
        text = generator.generate_purchase_order(i + 1)
        doc = Document(
            page_content=text,
            metadata={
                "document_type": "purchase_order",
                "document_id": f"PO-{i+1:06d}",
                "created_date": datetime.now().isoformat()
            }
        )
        documents.append(doc)
    
    # Generate sales orders
    for i in range(num_each):
        text = generator.generate_sales_order(i + 1)
        doc = Document(
            page_content=text,
            metadata={
                "document_type": "sales_order",
                "document_id": f"SO-{i+1:06d}",
                "created_date": datetime.now().isoformat()
            }
        )
        documents.append(doc)
    
    # Generate reports
    for i in range(num_each):
        text = generator.generate_report(i + 1)
        doc = Document(
            page_content=text,
            metadata={
                "document_type": "report",
                "document_id": f"REP-{i+1:06d}",
                "created_date": datetime.now().isoformat()
            }
        )
        documents.append(doc)
    
    print(f"✓ Generated {len(documents)} documents")
    print(f"  - Invoices: {num_each}")
    print(f"  - Purchase Orders: {num_each}")
    print(f"  - Sales Orders: {num_each}")
    print(f"  - Reports: {num_each}")
    
    return documents

# Create the dataset
raw_documents = create_document_dataset(num_each=25)

# Display a sample document
print("\n" + "="*80)
print("SAMPLE INVOICE:")
print("="*80)
print(raw_documents[0].page_content)
print("\nMetadata:", raw_documents[0].metadata)

## 3. Document Preprocessing and Chunking

In [ ]:
class SAPDocumentPreprocessor:
    """Preprocess and chunk SAP documents for optimal retrieval"""
    
    def __init__(self, chunk_size: int = 1000, chunk_overlap: int = 200):
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", " ", ""],
            length_function=len
        )
    
    def clean_text(self, text: str) -> str:
        """Clean and normalize document text"""
        # Remove extra whitespace
        text = " ".join(text.split())
        return text.strip()
    
    def process_documents(self, documents: List[Document]) -> List[Document]:
        """Process and chunk documents"""
        print(f"Processing {len(documents)} documents...")
        
        # Clean documents
        cleaned_docs = []
        for doc in documents:
            cleaned_content = self.clean_text(doc.page_content)
            cleaned_docs.append(
                Document(page_content=cleaned_content, metadata=doc.metadata)
            )
        
        # Split into chunks
        chunked_docs = self.text_splitter.split_documents(cleaned_docs)
        
        print(f"✓ Created {len(chunked_docs)} chunks from {len(documents)} documents")
        print(f"  Average chunks per document: {len(chunked_docs)/len(documents):.2f}")
        
        return chunked_docs

# Preprocess documents
preprocessor = SAPDocumentPreprocessor(
    chunk_size=config.CHUNK_SIZE,
    chunk_overlap=config.CHUNK_OVERLAP
)
processed_documents = preprocessor.process_documents(raw_documents)

print("\n" + "="*80)
print("SAMPLE PROCESSED CHUNK:")
print("="*80)
print(processed_documents[0].page_content[:500] + "...")
print("\nMetadata:", processed_documents[0].metadata)

## 4. Embedding Generation and Vector Store Setup

In [ ]:
# Initialize embedding model
print("Initializing embedding model...")
print(f"Using model: {config.EMBEDDING_MODEL}")

# Use open-source embeddings (no API key required)
embeddings = HuggingFaceEmbeddings(
    model_name=config.EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'},  # Use 'cuda' if GPU is available
    encode_kwargs={'normalize_embeddings': True}
)

print("✓ Embedding model loaded")

# Test embedding
test_text = "SAP Invoice INV-001234"
test_embedding = embeddings.embed_query(test_text)
print(f"  Embedding dimension: {len(test_embedding)}")
print(f"  Sample embedding (first 5 values): {test_embedding[:5]}")

In [ ]:
# Create vector store with ChromaDB
print("Creating vector store...")

# Create ChromaDB vector store
vector_store = Chroma.from_documents(
    documents=processed_documents,
    embedding=embeddings,
    collection_name=config.COLLECTION_NAME,
    persist_directory=config.VECTOR_STORE_PATH
)

print(f"✓ Vector store created with {len(processed_documents)} document chunks")
print(f"  Stored at: {config.VECTOR_STORE_PATH}")
print(f"  Collection: {config.COLLECTION_NAME}")

In [ ]:
# Test retrieval
print("Testing vector store retrieval...\n")

test_query = "What are the details of invoice INV-000001?"
retrieved_docs = vector_store.similarity_search(test_query, k=3)

print(f"Query: {test_query}")
print(f"\nRetrieved {len(retrieved_docs)} documents:\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"Document {i}:")
    print(f"Type: {doc.metadata.get('document_type', 'unknown')}")
    print(f"ID: {doc.metadata.get('document_id', 'unknown')}")
    print(f"Content preview: {doc.page_content[:200]}...")
    print("-" * 80)

## 5. RAG Pipeline Implementation

In [ ]:
# SAP-specific prompt template
SAP_PROMPT_TEMPLATE = """You are an intelligent SAP assistant specialized in answering questions about SAP documents including invoices, purchase orders, sales orders, and financial reports.

Use the following pieces of context from SAP documents to answer the question at the end. 
If you don't know the answer based on the provided context, say that you don't have enough information.
Always cite the document ID and type when referring to specific information.

Context:
{context}

Question: {question}

Helpful Answer (be specific and include document references):"""

SAP_PROMPT = PromptTemplate(
    template=SAP_PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

print("✓ Custom SAP prompt template created")

In [ ]:
class SAPRAGAgent:
    """RAG-based agent for SAP document queries"""
    
    def __init__(self, vector_store, embeddings, use_openai: bool = False):
        self.vector_store = vector_store
        self.embeddings = embeddings
        self.use_openai = use_openai
        
        # Initialize LLM
        if use_openai and config.OPENAI_API_KEY and config.OPENAI_API_KEY != "your-api-key-here":
            print("Initializing OpenAI LLM...")
            self.llm = ChatOpenAI(
                model_name=config.LLM_MODEL,
                temperature=config.TEMPERATURE,
                openai_api_key=config.OPENAI_API_KEY
            )
            print(f"✓ Using OpenAI model: {config.LLM_MODEL}")
        else:
            # Fallback to local/mock LLM for demonstration
            print("⚠ OpenAI API key not configured. Using mock LLM for demonstration.")
            print("  Set OPENAI_API_KEY environment variable to use real LLM.")
            self.llm = None
        
        # Create retriever
        self.retriever = self.vector_store.as_retriever(
            search_type="similarity",
            search_kwargs={"k": config.TOP_K_DOCUMENTS}
        )
        
        # Create QA chain if LLM is available
        if self.llm:
            self.qa_chain = RetrievalQA.from_chain_type(
                llm=self.llm,
                chain_type="stuff",
                retriever=self.retriever,
                return_source_documents=True,
                chain_type_kwargs={"prompt": SAP_PROMPT}
            )
    
    def query(self, question: str) -> Dict[str, Any]:
        """Query the SAP document system"""
        
        # Retrieve relevant documents
        retrieved_docs = self.retriever.get_relevant_documents(question)
        
        if self.llm:
            # Use full RAG pipeline with LLM
            result = self.qa_chain({"query": question})
            
            return {
                "question": question,
                "answer": result["result"],
                "source_documents": result["source_documents"],
                "num_sources": len(result["source_documents"])
            }
        else:
            # Mock response based on retrieved documents
            context = "\n\n".join([doc.page_content for doc in retrieved_docs[:3]])
            mock_answer = f"""Based on the retrieved SAP documents, here's what I found:

[Note: This is a mock response. Configure OpenAI API key for AI-generated answers]

Retrieved {len(retrieved_docs)} relevant documents:
{self._format_sources(retrieved_docs[:3])}

Context preview:
{context[:500]}...
"""
            return {
                "question": question,
                "answer": mock_answer,
                "source_documents": retrieved_docs,
                "num_sources": len(retrieved_docs)
            }
    
    def _format_sources(self, docs: List[Document]) -> str:
        """Format source documents for display"""
        sources = []
        for i, doc in enumerate(docs, 1):
            doc_type = doc.metadata.get('document_type', 'unknown')
            doc_id = doc.metadata.get('document_id', 'unknown')
            sources.append(f"{i}. {doc_type.upper()}: {doc_id}")
        return "\n".join(sources)
    
    def batch_query(self, questions: List[str]) -> List[Dict[str, Any]]:
        """Process multiple queries"""
        results = []
        for question in questions:
            result = self.query(question)
            results.append(result)
        return results

# Initialize RAG Agent
# Set use_openai=True if you have an API key configured
rag_agent = SAPRAGAgent(
    vector_store=vector_store,
    embeddings=embeddings,
    use_openai=False  # Change to True if you have OpenAI API key
)

print("\n✓ SAP RAG Agent initialized and ready")

## 6. Interactive Query Interface

In [ ]:
def display_query_result(result: Dict[str, Any]):
    """Display query results in a formatted manner"""
    print("\n" + "="*100)
    print("QUERY RESULT")
    print("="*100)
    print(f"\nQuestion: {result['question']}")
    print("\n" + "-"*100)
    print("Answer:")
    print("-"*100)
    print(result['answer'])
    print("\n" + "-"*100)
    print(f"Sources Used: {result['num_sources']} documents")
    print("-"*100)
    
    for i, doc in enumerate(result['source_documents'][:3], 1):
        print(f"\nSource {i}:")
        print(f"  Type: {doc.metadata.get('document_type', 'unknown')}")
        print(f"  ID: {doc.metadata.get('document_id', 'unknown')}")
        print(f"  Content: {doc.page_content[:300]}...")
    
    print("\n" + "="*100)

# Example queries
example_queries = [
    "What is the status of invoice INV-000001?",
    "Show me purchase orders for the IT department",
    "What are the total sales for Q1 2024?",
    "Which invoices are still outstanding?",
    "What is the delivery status of purchase order PO-000005?"
]

print("Example Queries Available:")
for i, q in enumerate(example_queries, 1):
    print(f"{i}. {q}")

In [ ]:
# Execute sample queries
print("\n" + "#"*100)
print("# EXECUTING SAMPLE QUERIES")
print("#"*100)

# Query 1: Invoice status
result1 = rag_agent.query(example_queries[0])
display_query_result(result1)

In [ ]:
# Query 2: Purchase orders by department
result2 = rag_agent.query(example_queries[1])
display_query_result(result2)

In [ ]:
# Custom query - Modify this cell to test your own questions
custom_question = "What are the payment terms for recent invoices?"

custom_result = rag_agent.query(custom_question)
display_query_result(custom_result)

## 7. Evaluation Metrics

In [ ]:
class RAGEvaluator:
    """Evaluate RAG system performance"""
    
    def __init__(self, rag_agent, embeddings):
        self.rag_agent = rag_agent
        self.embeddings = embeddings
    
    def evaluate_retrieval_quality(self, test_queries: List[Dict[str, Any]]) -> Dict[str, float]:
        """
        Evaluate retrieval quality
        test_queries format: [{"query": str, "expected_doc_type": str, "expected_doc_id": str}]
        """
        metrics = {
            "precision_at_k": [],
            "recall_at_k": [],
            "mrr": []  # Mean Reciprocal Rank
        }
        
        for test in test_queries:
            query = test["query"]
            expected_type = test.get("expected_doc_type")
            expected_id = test.get("expected_doc_id")
            
            # Retrieve documents
            retrieved = self.rag_agent.retriever.get_relevant_documents(query)
            
            # Check if expected document is in results
            relevant_found = 0
            rank = 0
            
            for i, doc in enumerate(retrieved, 1):
                if expected_type and doc.metadata.get("document_type") == expected_type:
                    relevant_found += 1
                    if rank == 0:
                        rank = i
                
                if expected_id and doc.metadata.get("document_id") == expected_id:
                    relevant_found += 1
                    if rank == 0:
                        rank = i
            
            # Calculate metrics
            k = len(retrieved)
            precision = relevant_found / k if k > 0 else 0
            recall = 1.0 if relevant_found > 0 else 0.0
            rr = 1.0 / rank if rank > 0 else 0.0
            
            metrics["precision_at_k"].append(precision)
            metrics["recall_at_k"].append(recall)
            metrics["mrr"].append(rr)
        
        # Average metrics
        return {
            "avg_precision@k": np.mean(metrics["precision_at_k"]),
            "avg_recall@k": np.mean(metrics["recall_at_k"]),
            "mean_reciprocal_rank": np.mean(metrics["mrr"]),
            "num_queries": len(test_queries)
        }
    
    def evaluate_answer_quality(self, qa_pairs: List[Dict[str, str]]) -> Dict[str, float]:
        """
        Evaluate answer quality using semantic similarity
        qa_pairs format: [{"question": str, "expected_answer": str}]
        """
        similarities = []
        
        for pair in qa_pairs:
            question = pair["question"]
            expected = pair["expected_answer"]
            
            # Get generated answer
            result = self.rag_agent.query(question)
            generated = result["answer"]
            
            # Calculate semantic similarity
            expected_emb = self.embeddings.embed_query(expected)
            generated_emb = self.embeddings.embed_query(generated)
            
            similarity = cosine_similarity(
                [expected_emb],
                [generated_emb]
            )[0][0]
            
            similarities.append(similarity)
        
        return {
            "avg_semantic_similarity": np.mean(similarities),
            "min_similarity": np.min(similarities),
            "max_similarity": np.max(similarities),
            "num_qa_pairs": len(qa_pairs)
        }
    
    def measure_latency(self, queries: List[str], num_runs: int = 3) -> Dict[str, float]:
        """Measure query latency"""
        import time
        
        latencies = []
        
        for query in queries:
            run_times = []
            for _ in range(num_runs):
                start = time.time()
                self.rag_agent.query(query)
                end = time.time()
                run_times.append(end - start)
            
            latencies.append(np.mean(run_times))
        
        return {
            "avg_latency_seconds": np.mean(latencies),
            "min_latency_seconds": np.min(latencies),
            "max_latency_seconds": np.max(latencies),
            "p95_latency_seconds": np.percentile(latencies, 95)
        }

# Initialize evaluator
evaluator = RAGEvaluator(rag_agent, embeddings)
print("✓ RAG Evaluator initialized")

In [ ]:
# Create test cases for evaluation
test_retrieval_queries = [
    {"query": "Show invoice INV-000001", "expected_doc_type": "invoice", "expected_doc_id": "INV-000001"},
    {"query": "Purchase order PO-000001 details", "expected_doc_type": "purchase_order", "expected_doc_id": "PO-000001"},
    {"query": "Sales order SO-000001", "expected_doc_type": "sales_order", "expected_doc_id": "SO-000001"},
    {"query": "Financial report REP-000001", "expected_doc_type": "report", "expected_doc_id": "REP-000001"},
]

# Evaluate retrieval quality
print("Evaluating retrieval quality...")
retrieval_metrics = evaluator.evaluate_retrieval_quality(test_retrieval_queries)

print("\n" + "="*80)
print("RETRIEVAL QUALITY METRICS")
print("="*80)
for metric, value in retrieval_metrics.items():
    if isinstance(value, float):
        print(f"{metric}: {value:.4f}")
    else:
        print(f"{metric}: {value}")

In [ ]:
# Measure query latency
print("\nMeasuring query latency...")
latency_queries = example_queries[:3]
latency_metrics = evaluator.measure_latency(latency_queries, num_runs=3)

print("\n" + "="*80)
print("LATENCY METRICS")
print("="*80)
for metric, value in latency_metrics.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
# Generate comprehensive evaluation report
def generate_evaluation_report(retrieval_metrics, latency_metrics):
    """Generate a comprehensive evaluation report"""
    report = f"""
{'='*100}
SAP RAG SYSTEM - EVALUATION REPORT
{'='*100}

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

SYSTEM CONFIGURATION:
  - Embedding Model: {config.EMBEDDING_MODEL}
  - Chunk Size: {config.CHUNK_SIZE}
  - Chunk Overlap: {config.CHUNK_OVERLAP}
  - Top-K Documents: {config.TOP_K_DOCUMENTS}
  - Total Documents: {len(raw_documents)}
  - Total Chunks: {len(processed_documents)}

RETRIEVAL PERFORMANCE:
  - Average Precision@{config.TOP_K_DOCUMENTS}: {retrieval_metrics['avg_precision@k']:.4f}
  - Average Recall@{config.TOP_K_DOCUMENTS}: {retrieval_metrics['avg_recall@k']:.4f}
  - Mean Reciprocal Rank: {retrieval_metrics['mean_reciprocal_rank']:.4f}
  - Test Queries: {retrieval_metrics['num_queries']}

LATENCY PERFORMANCE:
  - Average Query Time: {latency_metrics['avg_latency_seconds']:.4f}s
  - Min Query Time: {latency_metrics['min_latency_seconds']:.4f}s
  - Max Query Time: {latency_metrics['max_latency_seconds']:.4f}s
  - P95 Latency: {latency_metrics['p95_latency_seconds']:.4f}s

SYSTEM CAPABILITIES:
  ✓ Document retrieval and processing
  ✓ Semantic search across SAP documents
  ✓ Multi-document type support (invoices, POs, SOs, reports)
  ✓ Real-time query processing
  ✓ Source attribution and traceability

RECOMMENDATIONS:
  1. Fine-tune embedding model on SAP-specific vocabulary
  2. Implement hybrid search (keyword + semantic)
  3. Add query expansion for better recall
  4. Implement caching for frequently asked questions
  5. Add user feedback loop for continuous improvement

{'='*100}
"""
    return report

# Generate and display report
evaluation_report = generate_evaluation_report(retrieval_metrics, latency_metrics)
print(evaluation_report)

# Save report to file
with open("sap_rag_evaluation_report.txt", "w") as f:
    f.write(evaluation_report)
print("\n✓ Evaluation report saved to 'sap_rag_evaluation_report.txt'")

## 8. Advanced Features and Extensions

In [ ]:
class AdvancedSAPFeatures:
    """Advanced features for SAP RAG system"""
    
    def __init__(self, vector_store, embeddings):
        self.vector_store = vector_store
        self.embeddings = embeddings
    
    def filter_by_document_type(self, query: str, doc_type: str, k: int = 5) -> List[Document]:
        """Retrieve documents filtered by type"""
        all_docs = self.vector_store.similarity_search(query, k=k*3)
        filtered = [doc for doc in all_docs if doc.metadata.get("document_type") == doc_type]
        return filtered[:k]
    
    def filter_by_date_range(self, query: str, start_date: str, end_date: str, k: int = 5) -> List[Document]:
        """Retrieve documents within date range"""
        all_docs = self.vector_store.similarity_search(query, k=k*3)
        # In production, implement actual date filtering based on metadata
        return all_docs[:k]
    
    def aggregate_financial_data(self, doc_type: str = "report") -> Dict[str, Any]:
        """Aggregate financial data from reports"""
        # This is a simplified example
        # In production, parse actual numeric data from documents
        return {
            "total_revenue": "$X,XXX,XXX",
            "total_expenses": "$X,XXX,XXX",
            "net_profit": "$XXX,XXX",
            "note": "Implement actual parsing logic for production"
        }
    
    def get_document_statistics(self) -> Dict[str, Any]:
        """Get statistics about the document collection"""
        # This would query the vector store metadata
        doc_types = {}
        for doc in raw_documents:
            doc_type = doc.metadata.get("document_type", "unknown")
            doc_types[doc_type] = doc_types.get(doc_type, 0) + 1
        
        return {
            "total_documents": len(raw_documents),
            "total_chunks": len(processed_documents),
            "documents_by_type": doc_types,
            "avg_chunks_per_doc": len(processed_documents) / len(raw_documents)
        }
    
    def semantic_search_with_filters(self, query: str, filters: Dict[str, Any], k: int = 5) -> List[Document]:
        """Perform semantic search with metadata filters"""
        # Retrieve more documents than needed
        candidates = self.vector_store.similarity_search(query, k=k*5)
        
        # Apply filters
        filtered_docs = []
        for doc in candidates:
            match = True
            for key, value in filters.items():
                if doc.metadata.get(key) != value:
                    match = False
                    break
            if match:
                filtered_docs.append(doc)
            
            if len(filtered_docs) >= k:
                break
        
        return filtered_docs

# Initialize advanced features
advanced_features = AdvancedSAPFeatures(vector_store, embeddings)

# Test advanced features
print("\n" + "="*80)
print("ADVANCED FEATURES DEMO")
print("="*80)

# Get statistics
stats = advanced_features.get_document_statistics()
print("\nDocument Statistics:")
print(json.dumps(stats, indent=2))

# Filter by document type
print("\nFiltered Search (Invoices only):")
invoice_docs = advanced_features.filter_by_document_type(
    "payment terms",
    doc_type="invoice",
    k=3
)
for i, doc in enumerate(invoice_docs, 1):
    print(f"{i}. {doc.metadata['document_id']} - {doc.metadata['document_type']}")

print("\n✓ Advanced features demonstrated")

## 9. Production Deployment Considerations

In [ ]:
print("""
PRODUCTION DEPLOYMENT CHECKLIST:
================================

1. DATA & PREPROCESSING:
   □ Connect to real SAP system (HANA, S/4HANA, etc.)
   □ Implement data extraction pipelines
   □ Set up incremental updates for new documents
   □ Implement data validation and quality checks
   □ Handle different document formats (PDF, XML, JSON)

2. MODEL & EMBEDDINGS:
   □ Fine-tune embedding model on SAP domain data
   □ Evaluate different embedding models for your use case
   □ Implement model versioning and A/B testing
   □ Set up LLM fine-tuning pipeline if needed
   □ Configure API rate limits and quotas

3. VECTOR DATABASE:
   □ Choose production vector DB (Pinecone, Weaviate, Qdrant)
   □ Implement backup and recovery procedures
   □ Set up index optimization and maintenance
   □ Configure access controls and security
   □ Implement multi-tenancy if needed

4. RAG PIPELINE:
   □ Implement hybrid search (keyword + semantic)
   □ Add re-ranking for better results
   □ Implement query expansion and reformulation
   □ Add context compression for long documents
   □ Implement fallback strategies

5. MONITORING & EVALUATION:
   □ Set up logging and observability (LangSmith, etc.)
   □ Implement user feedback collection
   □ Track key metrics (latency, accuracy, user satisfaction)
   □ Set up alerts for system issues
   □ Regular model performance evaluation

6. SECURITY & COMPLIANCE:
   □ Implement authentication and authorization
   □ Encrypt sensitive data at rest and in transit
   □ Comply with data privacy regulations (GDPR, etc.)
   □ Implement audit logging
   □ Regular security assessments

7. SCALABILITY:
   □ Implement caching for common queries
   □ Set up load balancing
   □ Configure auto-scaling
   □ Optimize database queries and indexes
   □ Implement rate limiting

8. USER INTERFACE:
   □ Build web/mobile interface
   □ Implement chat history and session management
   □ Add document preview and download features
   □ Implement feedback buttons (helpful/not helpful)
   □ Support multi-language if needed

9. INTEGRATION:
   □ REST API for external applications
   □ Webhook support for real-time updates
   □ Integration with existing SAP tools
   □ SSO integration
   □ Export/import functionality

10. DOCUMENTATION & TRAINING:
    □ User documentation and guides
    □ API documentation
    □ Training materials for end users
    □ Maintenance and troubleshooting guides
    □ Regular updates and release notes

""")

## 10. Summary and Next Steps

In [ ]:
print(f"""
{'='*100}
SAP INTELLIGENT DOCUMENT PROCESSING - IMPLEMENTATION SUMMARY
{'='*100}

WHAT WE'VE BUILT:
✓ Complete RAG pipeline for SAP documents
✓ Document preprocessing and chunking
✓ Vector embeddings and similarity search
✓ Intelligent query answering
✓ Source attribution and traceability
✓ Evaluation metrics and monitoring
✓ Advanced filtering and search capabilities

DOCUMENTS PROCESSED:
  - Total Documents: {len(raw_documents)}
  - Document Chunks: {len(processed_documents)}
  - Supported Types: Invoices, Purchase Orders, Sales Orders, Reports

KEY FEATURES:
  1. Semantic Search: Find relevant documents using natural language
  2. Multi-Document Support: Handle various SAP document types
  3. Real-Time Queries: Fast retrieval and response generation
  4. Source Tracking: Every answer includes source documents
  5. Extensible: Easy to add new document types and features

NEXT STEPS:
  1. Connect to real SAP system data
  2. Configure OpenAI API key for production LLM
  3. Fine-tune models on your specific SAP terminology
  4. Implement web interface for end users
  5. Set up monitoring and feedback collection
  6. Deploy to production environment

EXAMPLE USAGE:
  result = rag_agent.query("What is the status of invoice INV-000001?")
  print(result['answer'])

CONFIGURATION:
  - Embedding Model: {config.EMBEDDING_MODEL}
  - LLM Model: {config.LLM_MODEL}
  - Vector Store: {config.VECTOR_STORE_PATH}
  - Top-K Results: {config.TOP_K_DOCUMENTS}

{'='*100}

For production deployment, refer to the deployment checklist above.
For questions or issues, consult the SAP and LangChain documentation.

Happy querying! 🚀
""")

## Appendix: Utility Functions

In [ ]:
# Utility functions for managing the RAG system

def save_vector_store():
    """Save the current vector store to disk"""
    # ChromaDB auto-persists with persist_directory
    print(f"✓ Vector store saved to {config.VECTOR_STORE_PATH}")

def load_existing_vector_store():
    """Load an existing vector store from disk"""
    if os.path.exists(config.VECTOR_STORE_PATH):
        loaded_store = Chroma(
            persist_directory=config.VECTOR_STORE_PATH,
            embedding_function=embeddings,
            collection_name=config.COLLECTION_NAME
        )
        print(f"✓ Loaded vector store from {config.VECTOR_STORE_PATH}")
        return loaded_store
    else:
        print(f"✗ No vector store found at {config.VECTOR_STORE_PATH}")
        return None

def add_new_documents(new_docs: List[Document]):
    """Add new documents to the existing vector store"""
    processed = preprocessor.process_documents(new_docs)
    vector_store.add_documents(processed)
    print(f"✓ Added {len(new_docs)} new documents ({len(processed)} chunks)")

def clear_vector_store():
    """Clear all documents from the vector store"""
    import shutil
    if os.path.exists(config.VECTOR_STORE_PATH):
        shutil.rmtree(config.VECTOR_STORE_PATH)
        print("✓ Vector store cleared")
    else:
        print("✗ No vector store to clear")

def export_qa_history(history: List[Dict], filename: str = "qa_history.json"):
    """Export Q&A history to JSON file"""
    with open(filename, 'w') as f:
        json.dump(history, f, indent=2)
    print(f"✓ Q&A history exported to {filename}")

print("✓ Utility functions loaded")